# Bayesian statistical analysis (all three models)

One **joint** hierarchical Beta regression (logit link) over up to **300** Macro-F1 cells
(3 models x 5 source domains x 5 target domains x 4 perturbation conditions).
It compares the model variants with each other (RQ1) and estimates the domain-shift and perturbation effects
(RQ2 to RQ5), so it needs the evaluation results of **base, large and lite**. It cannot be split per model.

**Prerequisite:** `run_base.ipynb`, `run_large.ipynb` and `run_lite.ipynb` finished with the **same `DRIVE_BASE`**.

Runs on CPU (no GPU needed; choose a standard runtime) and takes about 10 to 30 minutes at the final-report settings.

## 0. Configuration

In [ ]:
# ── EDIT ONLY THIS CELL ──────────────────────────────────────────────────────
REPO_URL   = "https://github.com/gredss/responsible-ai-toolkit.git"
BRANCH     = "main"

# Everything (checkpoints, outputs, results, statistics, logs) is saved directly inside DRIVE_BASE.
# All four notebooks must use the same DRIVE_BASE: they then share data_splits/,
# and the statistics notebook finds every model's results.
# To start a fresh experiment, point DRIVE_BASE to a new folder (existing files are reused, not overwritten).
USE_DRIVE  = True
DRIVE_BASE = "/content/drive/MyDrive/Colab Notebooks/thesis-grace"
SEED       = 42

# Final-report settings. For a quick test use draws=200, tune=200, chains=2.
MCMC       = dict(mcmc_draws=2000, mcmc_tune=2000, mcmc_chains=4)
PAIR_MODE  = "C"       # A: no pair effects | B: 25 pairs | C: 20 off-diagonal pairs (recommended)
ALLOW_PARTIAL = False  # True = analyse whatever models exist (model contrasts then need 2+ models)
# ─────────────────────────────────────────────────────────────────────────────

## 1. Environment

In [ ]:
import os, sys, json, time, shutil, subprocess, glob
os.environ["HF_HUB_DISABLE_XET"] = "1"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = DRIVE_BASE
else:
    ROOT = "/content/thesis_out"

REPO    = "/content/responsible-ai-toolkit"
CKPT    = f"{ROOT}/checkpoints"          # <CKPT>/<model>/<Domain>/best_model.pt
OUT     = f"{ROOT}/outputs"              # data_splits/ (shared by all models and by evaluation)
RESULTS = f"{ROOT}/results"              # <RESULTS>/<model>/complete_evaluation.json + perturbed CSVs
STATS   = f"{ROOT}/statistical_analysis"
LOGS    = f"{ROOT}/logs"
LOCAL_LOGS = "/content/_logs"            # logs are written locally, then copied to Drive (Drive is slow for streaming)
DOMAINS = ["Education", "Health", "Politics", "Sport", "Technology"]
ALL_MODELS = ["base", "large", "lite"]

for d in (CKPT, OUT, RESULTS, STATS, LOGS, LOCAL_LOGS):
    os.makedirs(d, exist_ok=True)

def eval_json(model): return f"{RESULTS}/{model}/complete_evaluation.json"
def ckpt_complete(model): return all(os.path.exists(f"{CKPT}/{model}/{d}/best_model.pt") for d in DOMAINS)

print("ROOT:", ROOT)

In [ ]:
# Clone (or refresh) the code from GitHub. Always uses the latest commit of BRANCH.
if os.path.isdir(f"{REPO}/.git"):
    subprocess.run(["git", "-C", REPO, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO], check=True)

print("Commit:", subprocess.run(["git", "-C", REPO, "log", "-1", "--format=%h %s"],
                                capture_output=True, text=True).stdout.strip())

# Guard: fail early if GitHub still holds the old code.
need = {"src/statistical_analyzer.py": "def run_full_analysis", "src/evaluate_pipeline.py": "--skip-statistics"}
for rel, marker in need.items():
    p = os.path.join(REPO, rel)
    if not os.path.exists(p) or marker not in open(p, encoding="utf-8").read():
        raise RuntimeError(f"{rel} in GitHub '{BRANCH}' is outdated (missing '{marker}'). "
                           "Commit and push your latest local changes, then re-run this cell.")
for rel in ("dataset/data/dict.json", "dataset/data/technology.csv"):
    assert os.path.exists(os.path.join(REPO, rel)), f"missing {rel} in the cloned repo"
print("Code check OK")

In [ ]:
# If Colab says it must restart the runtime after this install, restart and run from the top again.
%pip install -q "pymc==6.4.0" arviz pandas matplotlib

## 2. Load the evaluation results of every model

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, f"{REPO}/src")

status = {m: os.path.exists(eval_json(m)) for m in ALL_MODELS}
for m, ok in status.items():
    print(f"  {m:<6} {'FOUND  ' if ok else 'MISSING'} {eval_json(m)}")
missing = [m for m, ok in status.items() if not ok]
if missing and not ALLOW_PARTIAL:
    raise RuntimeError(f"Missing evaluation results for {missing}. Run run_{missing[0]}.ipynb first "
                       "(it must use the same DRIVE_BASE), or set ALLOW_PARTIAL = True.")
if len(status) - len(missing) < 1:
    raise RuntimeError("No evaluation results found.")

ALL_RESULTS = {m: json.load(open(eval_json(m), encoding="utf-8")) for m in ALL_MODELS if status[m]}
print("\nLoaded:", list(ALL_RESULTS))

In [ ]:
# Data check (no PyMC needed)
from statistical_analyzer import assemble_observations, validate_observations

obs_df = assemble_observations(ALL_RESULTS)
print("Observation rows:", len(obs_df), "(full design = 300)")
for k, v in validate_observations(obs_df).items():
    print(f"  {k}: (list omitted, see n_missing_combos)" if k == "missing_combinations" else f"  {k}: {v}")
display(obs_df.groupby(["model", "perturbation_condition"])["macro_f1"].agg(["count", "mean", "std"]).round(4))

## 3. Fit the model
The slow step. Output (trace, JSON, text report) is written to `<ROOT>/statistical_analysis/`.

In [ ]:
import pymc, arviz
from statistical_analyzer import StatisticalAnalyzer
print("PyMC", pymc.__version__, "| ArviZ", arviz.__version__)

analyzer = StatisticalAnalyzer(random_seed=SEED, output_dir=STATS, pair_mode=PAIR_MODE, **MCMC)
stat_results = analyzer.run_full_analysis(ALL_RESULTS)

print("\nStatus:", stat_results.get("status"))
if stat_results.get("status") == "failed":
    raise RuntimeError(f"Statistical analysis failed: {stat_results.get('reason')}")
print("Report:", stat_results.get("report_path"))

## 4. Results

In [ ]:
# MCMC convergence
conv = stat_results.get("convergence", {})
print("MCMC CONVERGENCE")
print(f"  R-hat max   : {conv.get('rhat_max', float('nan')):.4f}  (target <= 1.01)")
print(f"  Bulk-ESS min: {conv.get('ess_bulk_min', float('nan')):.0f}  (target >= 400)")
print(f"  Tail-ESS min: {conv.get('ess_tail_min', float('nan')):.0f}  (target >= 400)")
print(f"  Divergences : {conv.get('n_divergences', '?')}  (target 0)")
for w in conv.get("warnings", []):
    print("  WARNING:", w)

In [ ]:
# Posterior contrasts, grouped by research question
contrasts = stat_results.get("contrasts", {})
if not contrasts or "error" in contrasts:
    print("No contrasts:", contrasts.get("error", "unknown"))
else:
    df_c = pd.DataFrame([{"contrast": k, "mean": c["mean"], "ci_lower": c["ci_95_lower"], "ci_upper": c["ci_95_upper"],
                          "P(>0)": c["prob_positive"], "P(<0)": c["prob_negative"]}
                         for k, c in contrasts.items()]).set_index("contrast")
    groups = [("RQ1 model variants (positive = first model has higher F1)", df_c.index.str.startswith("model_")),
              ("RQ2 in-domain vs cross-domain (positive = in-domain higher)", df_c.index == "in_domain_vs_cross_domain"),
              ("RQ3 perturbation degradation (positive = F1 drops)",
               df_c.index.str.startswith("clean_vs_") | df_c.index.isin(["low_vs_medium", "medium_vs_high"])),
              ("RQ4 model x perturbation", df_c.index.str.startswith("interaction_degradation_")),
              ("RQ5 perturbation x scenario (negative = cross-domain degrades more)",
               df_c.index.str.startswith("interaction_scenario_"))]
    for title, mask in groups:
        if mask.any():
            print(f"\n{title}"); display(df_c[mask].round(4))

In [ ]:
# Descriptive comparison of the models (the raw numbers behind the contrasts)
rows = []
for m, r in ALL_RESULTS.items():
    in_f1 = [v["metrics"]["macro_f1"] for v in r["in_domain"].values()]
    cross = [v["metrics"]["macro_f1"] for k, v in r["cross_domain"].items() if k.split("->")[0] != k.split("->")[1]]
    row = {"model": m, "avg_in_domain_f1": sum(in_f1) / len(in_f1), "avg_cross_domain_f1": sum(cross) / len(cross)}
    for lv in ("low", "medium", "high"):
        row[f"avg_f1_{lv}"] = sum(b["semantic"][lv]["metrics"]["macro_f1"] for b in r["perturbation"].values()) / len(r["perturbation"])
    rows.append(row)
cmp_df = pd.DataFrame(rows).set_index("model")
display(cmp_df.round(4))
cmp_df.round(6).to_csv(f"{STATS}/model_comparison.csv")

In [ ]:
# Full text report
rp = stat_results.get("report_path", "")
print(open(rp, encoding="utf-8").read() if rp and os.path.exists(rp) else f"Report not found: {rp}")